# Data summary

Cleaning, label mapping, splits and leakage checks for the six experimental datasets.
All numbers are computed from the raw data by `src/data.py` and `src/splits.py`; no message text is printed.

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))  # run from notebooks/

import pandas as pd
from src.data import dedup_key, load_spam_email
from src.normalise import normalise
from src.splits import SEEDS, SPLITS_PATH, leakage_counts, load_all, remove_overlap, split_group

pd.set_option("display.width", 160)
datasets = {df["dataset"].iloc[0]: df for df in load_all()}
splits = pd.read_csv(SPLITS_PATH)

## 1. Cleaning

Rows removed by each cleaning step (`df.attrs["cleaning_log"]`). Steps run in this order: empty text, conflicting labels (every copy of a text labelled two ways), duplicates (lower-cased, white space collapsed).

In [2]:
pd.DataFrame({name: df.attrs["cleaning_log"] for name, df in datasets.items()}).T.fillna(0).astype(int)

,raw_rows,empty_text,conflicting_labels,duplicates,final_rows
mendeley,5971,0,68,106,5797
email_fraud,9396,3,0,1438,7955
bothbosu,1600,0,0,2,1598
uci,5574,0,0,415,5159
ceas_08,39154,2,0,1203,37949
spamassassin,5809,3,0,33,5773


## 2. Labels

`label_orig` counts, and how they map to `label_spam` (E4) and `label_fraud` (E3, E5). NA means the row is not used for that task.

In [3]:
rows = []
for name, df in datasets.items():
    for label, g in df.groupby("label_orig"):
        rows.append({"dataset": name, "label_orig": label, "rows": len(g),
                     "label_spam": "/".join(sorted(g["label_spam"].astype("string").fillna("NA").unique())),
                     "label_fraud": "/".join(sorted(g["label_fraud"].astype("string").fillna("NA").unique()))})
pd.DataFrame(rows)

,dataset,label_orig,rows,label_spam,label_fraud
0,mendeley,ham,4834,NA,0
1,mendeley,smishing,528,NA,1
2,mendeley,spam,435,NA,NA
3,email_fraud,fraud,4796,NA,1
4,email_fraud,legitimate,3159,NA,0
5,bothbosu,non-scam,799,NA,0/NA
6,bothbosu,scam,799,NA,1
7,uci,ham,4517,NA,NA
8,uci,spam,642,NA,NA
9,ceas_08,legitimate,17021,0,NA


In [4]:
bb = datasets["bothbosu"]
pd.crosstab(bb["subtype"], bb["label_fraud"].astype("string").fillna("NA"))

label_fraud,0,1,NA
subtype,,,
delivery,200,0,0
insurance,0,0,200
refund,0,200,0
reward,0,199,0
ssn,0,200,0
support,0,200,0
telemarketing,0,0,200
wrong,199,0,0


## 3. Split groups

Rows are split by group so that one group never appears in two splits. The group is the template, or for CEAS_08 its first 90 characters (to keep mass campaigns together).

In [5]:
pd.DataFrame([{"dataset": name, "rows": len(df), "templates": df["template"].nunique(),
               "split_groups": split_group(df).nunique(),
               "largest_group": int(split_group(df).value_counts().iloc[0])}
              for name, df in datasets.items()])

,dataset,rows,templates,split_groups,largest_group
0,mendeley,5797,5663,5663,7
1,email_fraud,7955,7662,7662,5
2,bothbosu,1598,1598,1598,1
3,uci,5159,5092,5092,6
4,ceas_08,37949,29862,26054,2930
5,spamassassin,5773,5611,5611,10


## 4. Split sizes

About 64 / 16 / 20 (train / val / test). BothBosu keeps its official test set, so its test split is the same for every seed.

In [6]:
size = splits.groupby(["dataset", "seed", "split"]).size().unstack("split")[["train", "val", "test"]]
share = size.div(size.sum(axis=1), axis=0).round(3).add_suffix("_share")
size.join(share)

split              train   val  test  train_share  val_share  test_share
dataset      seed                                                       
bothbosu     0      1023   256   319         0.64       0.16         0.2
             1      1023   256   319         0.64       0.16         0.2
             2      1023   256   319         0.64       0.16         0.2
ceas_08      0     24285  6075  7589         0.64       0.16         0.2
             1     24288  6072  7589         0.64       0.16         0.2
             2     24286  6071  7592         0.64       0.16         0.2
email_fraud  0      5091  1273  1591         0.64       0.16         0.2
             1      5091  1273  1591         0.64       0.16         0.2
             2      5091  1273  1591         0.64       0.16         0.2
mendeley     0      3709   928  1160         0.64       0.16         0.2
             1      3710   928  1159         0.64       0.16         0.2
             2      3710   928  1159         0.64       0.16         0.2
spamassassin 0      3694   924  1155         0.64       0.16         0.2
             1      3694   924  1155         0.64       0.16         0.2
             2      3695   924  1154         0.64       0.16         0.2
uci          0      3301   826  1032         0.64       0.16         0.2
             1      3301   826  1032         0.64       0.16         0.2
             2      3302   825  1032         0.64       0.16         0.2

## 5. Class balance per split

Positive share in each split, for the original labels and for each task after its NA rows are dropped (seed 0).

In [7]:
rows = []
for name, df in datasets.items():
    m = df.merge(splits[(splits["dataset"] == name) & (splits["seed"] == 0)][["id", "split"]], on="id")
    for col in ["label_orig", "label_fraud", "label_spam"]:
        sub = m[m[col].notna()]
        if sub.empty:
            continue
        dist = pd.crosstab(sub[col].astype("string"), sub["split"], normalize="columns").round(3)
        for label in dist.index:
            rows.append({"dataset": name, "column": col, "label": label,
                         **{s: dist.loc[label].get(s) for s in ["train", "val", "test"]},
                         "test_rows": int((sub["split"] == "test").sum())})
pd.DataFrame(rows)

,dataset,column,label,train,val,test,test_rows
0,mendeley,label_orig,ham,0.834,0.834,0.834,1160
1,mendeley,label_orig,smishing,0.091,0.091,0.091,1160
2,mendeley,label_orig,spam,0.075,0.075,0.075,1160
3,mendeley,label_fraud,0,0.901,0.902,0.901,1073
4,mendeley,label_fraud,1,0.099,0.098,0.099,1073
5,email_fraud,label_orig,fraud,0.603,0.603,0.603,1591
6,email_fraud,label_orig,legitimate,0.397,0.397,0.397,1591
7,email_fraud,label_fraud,0,0.397,0.397,0.397,1591
8,email_fraud,label_fraud,1,0.603,0.603,0.603,1591
9,bothbosu,label_orig,non-scam,0.500,0.500,0.502,319


## 6. Leakage checks

Number of split groups, templates and texts found in more than one split. All must be 0; `python -m src.splits` stops if any is not.

In [8]:
pd.DataFrame([{"dataset": name, "seed": seed,
               **leakage_counts(df, splits[(splits["dataset"] == name) & (splits["seed"] == seed)]
                                .set_index("id").loc[df["id"], "split"].set_axis(df.index))}
              for name, df in datasets.items() for seed in SEEDS])

,dataset,seed,shared_groups,shared_templates,shared_texts
0,mendeley,0,0,0,0
1,mendeley,1,0,0,0
2,mendeley,2,0,0,0
3,email_fraud,0,0,0,0
4,email_fraud,1,0,0,0
5,email_fraud,2,0,0,0
6,bothbosu,0,0,0,0
7,bothbosu,1,0,0,0
8,bothbosu,2,0,0,0
9,uci,0,0,0,0


Overlap of the test sets of two seeds, as a share of one test set. Two independent 20% samples would share about 20%. BothBosu shares 100% by design (official test set).

In [9]:
rows = []
for name in datasets:
    test = {s: set(splits[(splits["dataset"] == name) & (splits["seed"] == s) & (splits["split"] == "test")]["id"])
            for s in SEEDS}
    rows.append({"dataset": name, **{f"seed{a}-seed{b}": round(len(test[a] & test[b]) / len(test[a]), 3)
                                     for a, b in [(0, 1), (0, 2), (1, 2)]}})
pd.DataFrame(rows)

,dataset,seed0-seed1,seed0-seed2,seed1-seed2
0,mendeley,0.199,0.206,0.187
1,email_fraud,0.197,0.201,0.177
2,bothbosu,1.000,1.000,1.000
3,uci,0.195,0.196,0.207
4,ceas_08,0.126,0.118,0.193
5,spamassassin,0.195,0.198,0.197


## 7. Cross-dataset overlap (E3, E4)

Source rows that `remove_overlap` would drop, comparing the whole source with the whole target. E3 uses the fraud-task rows of each channel.

In [10]:
fraud = {n: datasets[n][datasets[n]["label_fraud"].notna()] for n in ["mendeley", "email_fraud", "bothbosu"]}
pairs = [(f"E3 {a} -> {b}", fraud[a], fraud[b]) for a in fraud for b in fraud if a != b]
pairs += [("E4 ceas_08 -> spamassassin", datasets["ceas_08"], datasets["spamassassin"]),
          ("E4 spamassassin -> ceas_08", datasets["spamassassin"], datasets["ceas_08"])]
pd.DataFrame([{"pair": p, "source_rows": len(s), **remove_overlap(s, t)[1]} for p, s, t in pairs])

,pair,source_rows,exact_text,template,removed
0,E3 mendeley -> email_fraud,5362,0,2,2
1,E3 mendeley -> bothbosu,5362,0,0,0
2,E3 email_fraud -> mendeley,7955,0,3,3
3,E3 email_fraud -> bothbosu,7955,0,0,0
4,E3 bothbosu -> mendeley,1198,0,0,0
5,E3 bothbosu -> email_fraud,1198,0,0,0
6,E4 ceas_08 -> spamassassin,37949,0,5,5
7,E4 spamassassin -> ceas_08,5773,0,4,4


## 8. Email artefact stripping (E4)

Emails containing "mailing list" or "listinfo", a corpus artefact (proposal 3.4), before and after `strip=True`. Both versions contain the same emails.

In [11]:
rows = []
for name in ["ceas_08", "spamassassin"]:
    for strip in (False, True):
        df = load_spam_email(name, strip=strip)
        cue = df["text"].str.lower().str.contains("mailing list|listinfo")
        rows.append({"dataset": name, "strip": strip, **df[cue]["label_orig"].value_counts().to_dict()})
pd.DataFrame(rows).fillna(0)

,dataset,strip,legitimate,spam
0,ceas_08,False,5107,11
1,ceas_08,True,847,10
2,spamassassin,False,2227,553
3,spamassassin,True,130,245


The CNN "THE DAILY TOP 10" spam campaign in CEAS_08 and where it falls in each seed.

In [12]:
ceas = datasets["ceas_08"]
cnn_ids = set(ceas.loc[ceas["template"].str.startswith("thedailytop"), "id"])
s = splits[(splits["dataset"] == "ceas_08") & splits["id"].isin(cnn_ids)]
cnn = s.groupby(["seed", "split"]).size().unstack(fill_value=0)
cnn["share_of_test"] = (cnn["test"] / splits[(splits["dataset"] == "ceas_08") & (splits["split"] == "test")]
                        .groupby("seed").size()).round(3)
cnn

split,test,train,share_of_test
seed,,,
0,2930,0,0.386
1,0,2930,0.000
2,0,2930,0.000


## 9. Text normalisation on clean text

Messages changed by `normalise()` in the three channels used for E5. Changes on clean text are mostly no-break spaces and obfuscation already present in fraud emails.

In [13]:
pd.DataFrame([{"dataset": n, "rows": len(df), "changed": int((df["text"].map(normalise) != df["text"]).sum()),
               **df[df["text"].map(normalise) != df["text"]]["label_orig"].value_counts().to_dict()}
              for n, df in datasets.items() if n in ("mendeley", "email_fraud", "bothbosu")]).fillna(0)

,dataset,rows,changed,ham,fraud,legitimate
0,mendeley,5797,3,3.0,0.0,0.0
1,email_fraud,7955,1374,0.0,1239.0,135.0
2,bothbosu,1598,0,0.0,0.0,0.0


## 10. Known limitations

| Item | Effect on the data |
|---|---|
| Header lines removed by `strip_artefacts` also cover spam that opens with fake headers | 9 spam emails (CEAS_08 4, SpamAssassin 5) |
| A `-- ` signature delimiter removes what follows | 62 spam emails (CEAS_08 2, SpamAssassin 60) |
| Some emails in two formats are not merged by the template | 6 pairs in the email fraud set |
| Short templates (only a URL or number) can match across datasets | 0 to 5 rows per pair in section 7 |
| `normalise()` leaves combining marks, circled letters and letters separated by two or more spaces | 7, 2 and 0 messages in the three channels |
| `normalise()` also joins real runs of four or more single letters | 3 Mendeley messages |
| BothBosu is generated by a language model; paraphrased dialogues are not caught by the template | not measured |
| The CNN campaign is in the seed 0 test split (section 8) | about 39% of that test split |